# C05 · Boot from reset - linker script, startup and memory map

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/01_c_bare_metal/C05_boot_from_reset_linker_script_startup_and_memory.ipynb)

**Track:** C and bare-metal · **Skill axis (sheet):** C and bare-metal · **Time:** about 70 min

**Skill Hub concepts:** Linker script sections; Load versus run address; Reset sequence and vector table; Stack and heap layout; Clock gating

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("C05")

## Map: from power-on to `main`

```
 FLASH 0x0000_0000                        RAM 0x2000_0000                       _estack = 0x2001_0000
 +-------------+---------+----------+      +--------+--------+--------> heap   <-------- stack +
 | .isr_vector |  .text  | .data    | ---> | .data  |  .bss  |  _end              (grows down)  |
 | SP, Reset.. | .rodata | (initial | copy | (run   | zeroed |                                 |
 +-------------+---------+  values) |      |  copy) |        |                                 |
                         ^ LMA      |      ^ VMA                                               |
                         _sidata             _sdata.._edata  _sbss.._ebss
```

| Step | Who does it | What happens |
|---|---|---|
| 1 | hardware | loads SP from vector table word 0 and PC from word 1 (address with bit 0 = 1 for Thumb) |
| 2 | `Reset_Handler` | copies `.data` from its load address (flash) to its run address (RAM) |
| 3 | `Reset_Handler` | zeroes `.bss` |
| 4 | `SystemInit` (real chips) | clock tree: PLL, flash wait states; peripheral clocks stay gated |
| 5 | your driver | enables the peripheral clock (for example `RCC->AHB2ENR`) **before** touching its registers |
| 6 | `main` | runs; heap grows up from `_end`, stack grows down from `_estack` |

Linker script pieces you need: `MEMORY` regions, `> RAM AT> FLASH` (run address in RAM, load address in flash),
`LOADADDR(.data)`, `KEEP(*(.isr_vector))`, `(NOLOAD)` for `.bss`, and symbols the startup code reads.

Target: QEMU `mps2-an385` (Cortex-M3). Treat `0x0000_0000` (256 KB) as flash and `0x2000_0000` (64 KB) as RAM.
The checker fills RAM with `0xA5` garbage before reset, so a missing copy or zero loop shows up immediately.

In [ ]:
from roboprep import clab
clab.ensure_arm()

## Task 1: the linker script

In [ ]:
%%writefile link.ld
ENTRY(Reset_Handler)
MEMORY {
/* TODO: FLASH and RAM regions */
}
_estack = ORIGIN(RAM) + LENGTH(RAM);
SECTIONS {
  .isr_vector : { KEEP(*(.isr_vector)) } > FLASH
  .text : { *(.text*) *(.rodata*) . = ALIGN(4); } > FLASH
/* TODO: _sidata, then .data (run in RAM, load from FLASH) with _sdata/_edata,
         then .bss with _sbss/_ebss. Keep everything 4-byte aligned. */
  _end = .;
}

## Task 2: vector table and `Reset_Handler`

In [ ]:
%%writefile startup.c
#include <stdint.h>
extern uint32_t _sidata, _sdata, _edata, _sbss, _ebss, _estack;
int main(void);

void Reset_Handler(void) {
    /* TODO: copy .data from flash to RAM, then zero .bss */
    main();
    for (;;) {}
}
void Default_Handler(void) { for (;;) {} }

typedef void (*vec_t)(void);
__attribute__((section(".isr_vector"), used))
const vec_t vector_table[16] = {
    0, 0,   /* TODO: word 0 = initial stack pointer, word 1 = reset handler */
    Default_Handler, Default_Handler, Default_Handler, Default_Handler, Default_Handler,
    0, 0, 0, 0, Default_Handler, Default_Handler, 0, Default_Handler, Default_Handler
};

In [ ]:
%%writefile main.c
#include <stdint.h>
#include "semi.h"           /* Arm semihosting: prints through the debugger (QEMU here) */
uint32_t initialized = 0xC0FFEE42u;   /* .data */
uint32_t zeroed;                      /* .bss  */
int main(void) {
    sh_puts("booted\n");
    sh_puts(initialized == 0xC0FFEE42u ? "data_ok\n" : "data_BAD\n");
    sh_puts(zeroed == 0u ? "bss_ok\n" : "bss_BAD\n");
    sh_exit(0);
    return 0;
}

In [ ]:
lab.check("boot")

Read what the linker produced: section addresses (VMA = run, LMA = load) and the symbol order.

In [ ]:
print(clab.arm_tool("arm-none-eabi-objdump -h fw.elf | sed -n '5,14p'"))
print(clab.arm_tool("arm-none-eabi-nm -n fw.elf | grep -E ' (_s|_e|_end|vector_table|Reset_Handler|initialized|zeroed)'"))

## Checkpoint: Math
**M1 (Apply).** Map file: `_sidata = 0x0000_0F00`, `_sdata = 0x2000_0000`, `_edata = 0x2000_0020`. How many bytes
does `Reset_Handler` copy?

**M2 (Apply).** Byte offset of the vector for external interrupt IRQ5 (16 system entries come first, 4 bytes each)?

**M3 (Understand).** `Reset_Handler` is linked at `0x0000_0100`. What value is stored in vector word 1?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** On a real STM32 you write `.data : {...} > RAM` and forget `AT> FLASH`. Effect after power-up?
- a) link error
- b) initialized globals hold garbage: their initial values were never placed in flash to copy from
- c) nothing; RAM keeps values when power is off

**K2 (Understand).** Walk through reset to `main` on an STM32G474, including where clock gating appears and what
happens if you write a GPIO register before enabling its clock.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `c05.h`).
3. In `roboprep/checks/c05.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()